# Вариант 1 — Модуль `m01` (`extract_metadata`): запуск и проверка в Google Colab

Блокнот прогоняет студенческий модуль [modules/m01_metadata.py](../../modules/m01_metadata.py) на всех трёх ступенях (`rules`, `model`, `llm`), проверяет контракт (`check_module`, `tools/merge_and_check.py`), строит RDF-граф и проверяет SPARQL-запрос `papers_by_author`.

Режимы работы (переключаются в **Ячейке 1**):
- `USE_DRIVE = False` — быстрый запуск прямо сейчас: скачивается тестовая статья `arXiv:2611.16131v1` в `work/pdf/`.
- `USE_DRIVE = True` — боевой прогон на Google Drive с `practice2026/` и `kg_data/test_set/`.

Перед запуском в Colab желательно включить GPU: **Среда выполнения → Сменить среду выполнения → T4 GPU**.

## 1. Подключение репозитория и библиотек

In [ ]:
import os, sys, json

USE_DRIVE = False  # True — подключить Google Drive (practice2026/ и kg_data/test_set/)
REPO_URL = 'https://github.com/fake-faq/muiv3pr.git'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    ROOT = '/content/muiv3pr'
    if not os.path.exists(ROOT):
        !git clone -q {REPO_URL} {ROOT}
    else:
        !git -C {ROOT} pull --ff-only
    !pip install -q -r {ROOT}/requirements.txt
    !pip install -q transformers accelerate sentencepiece
else:
    cand = [os.path.abspath('.'), os.path.abspath('..'), os.path.abspath(os.path.join('..', '..'))]
    ROOT = next(p for p in cand if os.path.isdir(os.path.join(p, 'mathkg')))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/practice2026'
    TEST = '/content/drive/MyDrive/kg_data/test_set'
else:
    WORK = os.path.join(ROOT, 'work')
    TEST = None

os.makedirs(WORK, exist_ok=True)
FACTS_BASE = os.path.join(WORK, 'facts_base')
FACTS_DIR = os.path.join(WORK, 'facts_m01')
ONTOLOGY = [os.path.join(ROOT, 'ontology', n) for n in ('MocassinPlus_v2.ttl', 'SciLibReco.ttl')]

import pandas as pd
pd.set_option('display.max_colwidth', 120)
print('ROOT:', ROOT)
print('WORK:', WORK)
print('TEST:', TEST)

## 2. Набор статей и прогон общей основы `base`

In [ ]:
import csv, glob, urllib.request
from mathkg.base import build_corpus

if TEST and os.path.isdir(os.path.join(TEST, 'base')):
    BASE_DIR = os.path.join(TEST, 'base')
    print('Используется общий тестовый набор:', BASE_DIR)
elif USE_DRIVE and os.path.isdir(os.path.join(WORK, 'pdf')):
    BASE_DIR = os.path.join(WORK, 'base')
    stats = build_corpus(os.path.join(WORK, 'index.csv'), os.path.join(WORK, 'pdf'), BASE_DIR, log_every=50)
    print('build_corpus:', stats)
else:
    PDF_DIR = os.path.join(WORK, 'pdf')
    BASE_DIR = os.path.join(WORK, 'base')
    os.makedirs(PDF_DIR, exist_ok=True)
    sample_pdf = os.path.join(PDF_DIR, 'A0001.pdf')
    if not os.path.exists(sample_pdf):
        req = urllib.request.Request('https://arxiv.org/pdf/2611.16131v1', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=120) as resp, open(sample_pdf, 'wb') as out:
            out.write(resp.read())
    own = sorted(os.path.basename(p) for p in glob.glob(os.path.join(PDF_DIR, '*.pdf')) if not p.endswith('A0001.pdf'))
    rows = [('A0001', 'A0001.pdf')] + [(f'U{k:04d}', name) for k, name in enumerate(own, 1)]
    index_csv = os.path.join(WORK, 'index.csv')
    with open(index_csv, 'w', newline='', encoding='utf-8') as f:
        csv.writer(f).writerows([('doc_id', 'file')] + rows)
    stats = build_corpus(index_csv, PDF_DIR, BASE_DIR, log_every=20)
    print('build_corpus:', stats)

docs_df = pd.read_json(os.path.join(BASE_DIR, 'docs.jsonl'), lines=True)
ok_ids = docs_df[docs_df.status == 'ok'].doc_id.tolist()
print('Статусы набора:', docs_df['status'].value_counts().to_dict())
print('Пригодных статей (ok):', len(ok_ids), ok_ids[:10])

## 3. Этапы 1–2: статистика первой страницы и строки `first_page_lines`

In [ ]:
import re
from mathkg.base import read_doc

checks = {
    'abstract_label': re.compile(r'(Аннотация|Abstract|Резюме|Summary)', re.I),
    'udc': re.compile(r'(УДК|UDC)\s*:?\s*\d', re.I),
    'msc': re.compile(r'(MSC|Mathematics Subject Classification|AMS subject classification)', re.I),
    'doi': re.compile(r'10\.\d{4,9}/'),
    'email': re.compile(r'[\w.+-]+@[\w-]+(?:\.[\w-]+)+'),
}
counts = {k: 0 for k in checks}
for doc_id in ok_ids:
    d = read_doc(BASE_DIR, doc_id)
    head = d.text[:d.pages[0].end] if d.pages else d.text[:5000]
    for k, rx in checks.items():
        if rx.search(head):
            counts[k] += 1

print('Статистика 1-й страницы (статей:', len(ok_ids), '):', counts)
d0 = read_doc(BASE_DIR, ok_ids[0])
print(f'\nПервые 15 строк first_page_lines для {d0.doc_id} (body_size={d0.meta.get("quality", {}).get("body_size")}):')
for ln in d0.first_page_lines[:15]:
    print(f"  size={ln['size']:4.1f} bold={str(ln['bold']):5s} italic={str(ln['italic']):5s} | {ln['text'][:90]}")

## 4. Базовая версия vs студенческий `m01` (ступень `rules`)

In [ ]:
from mathkg import pipeline, registry

# Базовая версия из mathkg/modules/m01_metadata.py
pipeline.setup()
st_base = pipeline.run_corpus(BASE_DIR, FACTS_BASE, names=['extract_metadata'], redo=['extract_metadata'])

# Версия студента из modules/m01_metadata.py — ступень rules
os.environ['MATHKG_M01'] = 'rules'
pipeline.setup(student_modules=os.path.join(ROOT, 'modules'))
print('Активный автор модуля:', registry.REGISTRY['extract_metadata'].author)
st_rules = pipeline.run_corpus(BASE_DIR, FACTS_DIR, names=['extract_metadata'], redo=['extract_metadata'])

display(pd.DataFrame([
    dict(version='base', **st_base['extract_metadata']),
    dict(version='m01_rules', **st_rules['extract_metadata']),
]))

f_rules = pipeline.load_facts(FACTS_DIR, 'extract_metadata', ok_ids[0])
print('Фрагменты (m01_rules):', [(f.id, f.cls, f.method) for f in f_rules.fragments])
for r in f_rules.relations[:25]:
    print(f'  [{r.method}] {r.subj.split("/")[-1]} -> {r.pred} -> {str(r.obj)[:90]}')

## 5. Ступень `model`: готовый NER `Babelscape/wikineural-multilingual-ner`

In [ ]:
os.environ['MATHKG_M01'] = 'model'
os.environ['MATHKG_M01_THR'] = '0.7'
pipeline.setup(student_modules=os.path.join(ROOT, 'modules'))
st_model = pipeline.run_corpus(BASE_DIR, FACTS_DIR, names=['extract_metadata'], redo=['extract_metadata'])
print('m01_model:', st_model['extract_metadata'])

for doc_id in ok_ids[:5]:
    fx = pipeline.load_facts(FACTS_DIR, 'extract_metadata', doc_id)
    mod_rels = [(r.pred, r.obj, r.confidence) for r in fx.relations if r.method == 'model']
    if mod_rels:
        print(doc_id, 'находки NER:', mod_rels)

## 6. Ступень `llm`: выборочный `Qwen/Qwen2.5-1.5B-Instruct` с кэшем

In [ ]:
# Вызывается только если правила + NER не нашли журнал, авторов или организации (при наличии e-mail)
os.environ['MATHKG_M01'] = 'llm'
os.environ['MATHKG_M01_CACHE'] = os.path.join(WORK, 'm01_llm_cache.json')
pipeline.setup(student_modules=os.path.join(ROOT, 'modules'))
st_llm = pipeline.run_corpus(BASE_DIR, FACTS_DIR, names=['extract_metadata'], redo=['extract_metadata'])
print('m01_llm:', st_llm['extract_metadata'])

cache_file = os.environ['MATHKG_M01_CACHE']
if os.path.exists(cache_file):
    with open(cache_file, encoding='utf-8') as fh:
        print('Записей в дисковом кэше LLM:', len(json.load(fh)))

for doc_id in ok_ids[:5]:
    fx = pipeline.load_facts(FACTS_DIR, 'extract_metadata', doc_id)
    llm_rels = [(r.pred, r.obj, r.confidence) for r in fx.relations if r.method == 'llm']
    if llm_rels:
        print(doc_id, 'находки LLM:', llm_rels)

## 7. Проверка контракта (`check_module`, `merge_and_check`) и оценка по эталону

In [ ]:
from mathkg.check import check_module
from mathkg.evaluate import evaluate

rep = check_module('extract_metadata', BASE_DIR, FACTS_DIR, ONTOLOGY, limit_ms=2000)
print('check_module passed:', rep['passed'], '| ms_per_doc:', rep['ms_per_doc'])
print('errors:', rep['errors'][:10])
print('counts:', rep['counts'])

!python {ROOT}/tools/merge_and_check.py check {FACTS_DIR}

gold_path = os.path.join(ROOT, 'gold', 'm01.jsonl')
if os.path.exists(gold_path):
    ev = evaluate('extract_metadata', gold_path, BASE_DIR, FACTS_DIR)
    display(pd.DataFrame(ev['by_key']).T)
    print('пропуски:', ev['missed'][:5])
    print('лишние:', ev['extra'][:5])
else:
    print('Файл эталона gold/m01.jsonl пока не создан — разметка выполняется на Этапе 3.')

## 8. Граф RDF, склейка графов и SPARQL-запрос `papers_by_author`

In [ ]:
import rdflib
from mathkg.to_rdf import corpus_to_ntriples

graph_nt = os.path.join(WORK, 'graph_m01.nt')
merged_nt = os.path.join(WORK, 'merged.nt')
n_triples = corpus_to_ntriples(BASE_DIR, FACTS_DIR, graph_nt, student='student-01', names=['extract_metadata'])
print('Записано троек:', n_triples, '->', graph_nt)

!python {ROOT}/tools/merge_and_check.py merge {merged_nt} {graph_nt} --mode sameas

g = rdflib.Graph()
g.parse(merged_nt, format='nt')
q = '''
PREFIX scilibreco: <https://scilib.ai/ontology/scilibreco#>
SELECT ?doc ?title ?year (GROUP_CONCAT(DISTINCT ?authorName; SEPARATOR='; ') AS ?authors)
WHERE {
  ?doc scilibreco:hasAuthor ?author .
  ?author scilibreco:hasName ?authorName .
  OPTIONAL { ?doc scilibreco:hasTitle ?title }
  OPTIONAL { ?doc scilibreco:hasPublicationYear ?year }
}
GROUP BY ?doc ?title ?year
LIMIT 20
'''
res = g.query(q)
display(pd.DataFrame([[str(x) if x is not None else '' for x in row] for row in res],
                     columns=[str(v) for v in res.vars]))